# Step 4: SVD++ — true collaborative filtering

Your bias model scored RMSE 0.8998 on hidden data. This is now the
number to beat.

**What's different about SVD++:** the bias model just averages —
it has no idea that user A and user B have similar taste, or that
movie X and movie Y appeal to the same kind of person. SVD++ learns
this. It assigns every user a vector of (say) 50 numbers, and every
movie a vector of 50 numbers — these aren't hand-picked, they're
learned by an optimization process that tries to make
`user_vector · movie_vector` reconstruct every known rating as
closely as possible.

Once trained, predicting an unseen `(user, movie)` pair is just:
combine that user's learned vector with that movie's learned vector.
No averaging — the model has inferred something like "this user
tends to like atmospheric sci-fi" and "this movie IS atmospheric
sci-fi" purely from rating patterns, without ever being told either
fact directly.

We validate it with the exact same hide-and-predict pattern as
before — nothing about the validation method changes, only the
model does.

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from surprise import SVDpp, Dataset, Reader

DATA_DIR = "data"
N_USERS_TO_KEEP = 15_000   # same user-based subsampling as the bias model step

train_full = pd.read_csv(f"{DATA_DIR}/train.csv")
print(f"Full data: {train_full.shape}")

Full data: (10000038, 4)


In [ ]:
# Subsample the training data to keep only a subset of users for faster tuning
# We use a random number generator with a fixed seed for reproducibility, and we select a random subset of users to keep in the training data. 
# The number of users to keep is determined by the N_USERS_TO_KEEP constant, but we ensure that we don't try to keep more users than exist in the dataset.
rng = np.random.default_rng(42)
all_users = train_full["userId"].unique()
n_to_keep = min(N_USERS_TO_KEEP, len(all_users))
chosen_users = rng.choice(all_users, size=n_to_keep, replace=False)

train = train_full[train_full["userId"].isin(chosen_users)].reset_index(drop=True)
print(f"Subsample: {train.shape}, users: {train['userId'].nunique():,}")

Subsample: (930202, 4), users: 15,000


In [ ]:
# Split the subsampled training data into visible and hidden sets for model training and evaluation. 
# The visible set will be used to train the model, while the hidden set will be used to evaluate its performance. 
# We also check how many rows in the hidden set contain users or movies that were not seen in the visible set, which can affect the model's ability to make accurate predictions.

visible, hidden = train_test_split(train, test_size=0.2, random_state=42)

unseen_users  = (~hidden["userId"].isin(visible["userId"])).sum()
unseen_movies = (~hidden["movieId"].isin(visible["movieId"])).sum()

print(f"Visible: {len(visible):,}   Hidden: {len(hidden):,}")
print(f"Hidden rows with never-before-seen user:  {unseen_users} ({unseen_users/len(hidden):.2%})")
print(f"Hidden rows with never-before-seen movie: {unseen_movies} ({unseen_movies/len(hidden):.2%})")

Visible: 744,161   Hidden: 186,041
Hidden rows with never-before-seen user:  0 (0.00%)
Hidden rows with never-before-seen movie: 1670 (0.90%)


## Train SVD++ on the visible set only

A couple of settings worth knowing about:
- `n_factors` — how many numbers are in each user/movie vector.
  More factors = richer model, but slower and more prone to overfitting
  on a small dataset. 50 is a reasonable starting point.
- `n_epochs` — how many passes the optimizer makes over the data.
  More epochs = better fit, up to a point, but takes longer.

This cell prints how long training takes — useful context since SVD++
is much slower than the bias model, especially as you scale up to more
users.

In [ ]:
# Train the SVD++ model on the visible data and measure the training time. 
# We use the Surprise library's SVDpp implementation, which is an extension of the SVD algorithm that incorporates implicit feedback. 
# The model is trained with a specified number of latent factors and epochs, and we set a random seed for reproducibility.
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(visible[["userId", "movieId", "rating"]], reader)
trainset = data.build_full_trainset()

t0 = time.time()
model = SVDpp(n_factors=50, n_epochs=20, random_state=42)
model.fit(trainset)
print(f"Training took {time.time() - t0:.1f}s")

Training took 660.3s


## Predict on the hidden set

This loops over every hidden row and asks the model: based on what
you learned about this user and this movie from the visible data,
what rating do you predict? The model was never shown these exact
(user, movie) combinations during training.

In [ ]:
# Predict the ratings for the hidden set using the trained model and measure the prediction time.
# We create a copy of the hidden set and apply the model's predict method to each row, which returns an estimated rating (est) for the user-movie pair.
t0 = time.time()
hidden = hidden.copy()
hidden["predicted"] = hidden.apply(
    lambda r: model.predict(r["userId"], r["movieId"]).est, axis=1
)
hidden["predicted"] = hidden["predicted"].clip(0.5, 5.0)
print(f"Predicting took {time.time() - t0:.1f}s")

hidden[["userId", "movieId", "rating", "predicted"]].head()

Predicting took 48.9s


,userId,movieId,rating,predicted
601100,161675,163072,4.5,2.880815
758629,51708,551,5.0,4.423056
378850,118951,412,4.0,3.746940
743741,70145,63876,3.5,3.618830
490140,49457,111781,5.0,3.792682


## Check RMSE — does true collaborative filtering beat the bias model?

In [ ]:
# Compute the Root Mean Squared Error (RMSE) between the actual ratings and the predicted ratings for the hidden set.
# RMSE is a common metric for evaluating the accuracy of predicted ratings in recommendation systems, with lower values indicating better performance.
rmse_svdpp = np.sqrt(mean_squared_error(hidden["rating"], hidden["predicted"]))

print(f"RMSE — user + movie bias (previous step) : 0.8998")
print(f"RMSE — SVD++ (this step)                 : {rmse_svdpp:.4f}")

RMSE — user + movie bias (previous step) : 0.8998
RMSE — SVD++ (this step)                 : 0.8617


## What to expect, and what's next

SVD++ should outperform the bias model, typically by a modest but
real margin, bias models already capture a lot of the signal, so
SVD++'s gain often looks smaller proportionally than the bias model's
gain over the flat mean was. That's normal; we're past the point of
huge jumps and into the territory of careful refinement.

From here, the next steps are:
1. **Scale up** — re-run with more users (or the full dataset) since
   SVD++ tends to improve further with more data to learn from.
2. **Tune hyperparameters** — try different `n_factors` / `n_epochs`
   and see which combination gives the best hidden-data RMSE.
3. **Bring in content-based features** — genome scores, IMDB metadata
   — to build a second model that complements SVD++, then blend the two.

We'll do these one at a time, validating with the same hide-and-predict
check at every step.